# Text-to-SQL QLoRA on a free T4

Qwen2.5-Coder-1.5B-Instruct fine-tuned on the validated SQLite dataset from this repo's ETL, then scored by running the generated SQL. Needs a T4 and two Colab secrets: `HF_TOKEN` (write) and `WANDB_API_KEY`.

## 0. Runtime check

In [1]:
import torch, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"],
                     capture_output=True, text=True).stdout)
gpu = torch.cuda.get_device_name(0)
assert "T4" in gpu, f"expected a T4, got {gpu}"
print("torch", torch.__version__, "| cuda", torch.version.cuda)

name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 0 MiB

torch 2.14.1+cu130 | cuda 13.0


## 1. Install
Unsloth's Colab recipe for Qwen2.5-Coder. The transformers/trl pins matter.

In [2]:
%%capture
import os
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2
!pip install wandb

In [3]:
!pip freeze | grep -E "unsloth|trl|transformers|peft|bitsandbytes|torch|datasets"

bitsandbytes==0.50.2
causal_conv1d @ https://github.com/unslothai/unsloth/releases/download/prebuilt-wheels-cu13/causal_conv1d-1.7.0+cu13torch2.14cxx11abiTRUE-cp313-cp313-linux_x86_64.whl
datasets==4.3.0
mamba_ssm @ https://github.com/unslothai/unsloth/releases/download/prebuilt-wheels-cu13/mamba_ssm-2.3.2.post1+cu13torch2.14cxx11abiTRUE-cp313-cp313-linux_x86_64.whl
peft==0.21.2
sentence-transformers==5.7.0
tensorflow-datasets==4.9.10
torch==2.14.1+cu130
torchao==0.10.0
torchaudio==2.11.0+cu130
torchcodec==0.17.0+cu130
torchdata==0.11.0
torchsummary==1.5.1
torchtune==0.6.1
torchvision==0.29.1+cu130
transformers==4.57.6
trl==0.22.2
unsloth==2026.11.4
unsloth_zoo==2026.11.4
vega-datasets==0.9.0
xformers @ https://download.pytorch.org/whl/cu130/xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl


## 2. Secrets

In [4]:
import os
from google.colab import userdata
from huggingface_hub import login, whoami

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
login(token=os.environ["HF_TOKEN"])
HF_USER = whoami()["name"]
print("logged in as", HF_USER)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


logged in as parthasaradhi789


## 3. Config
Same values as `configs/train.yaml`. The dataset repo is the one pushed by `make push-dataset`.

In [5]:
import subprocess

CFG = dict(
    base_model="unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    max_seq_length=2048,
    lora_r=16, lora_alpha=32, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    epochs=1, batch_size=8, grad_accum=2,
    learning_rate=2e-4, lr_scheduler="cosine", warmup_ratio=0.03,
    optimizer="adamw_8bit", weight_decay=0.01,
    eval_steps=100, save_steps=200, seed=3407,
)
DATASET_REPO = f"{HF_USER}/text2sql-sqlite-clean"
OUT_REPO_LORA = f"{HF_USER}/text2sql-qwen2.5-coder-1.5b-lora"
OUT_REPO_GGUF = f"{HF_USER}/text2sql-qwen2.5-coder-1.5b-GGUF"
REPO_URL = "https://github.com/wittyparth/text2sql-specialist"  # must be public for this clone

# the repo gives us the same prompt/schema/sqlexec code the eval and the app use
!rm -rf repo && git clone --depth 1 {REPO_URL} repo
GIT_SHA = subprocess.run(["git", "-C", "repo", "rev-parse", "--short", "HEAD"],
                         capture_output=True, text=True).stdout.strip()
import sys; sys.path.insert(0, "repo/src")
print(DATASET_REPO, "|", OUT_REPO_LORA, "|", OUT_REPO_GGUF, "| repo", GIT_SHA)

Cloning into 'repo'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 128 (delta 0), reused 116 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 856.60 KiB | 12.60 MiB/s, done.
parthasaradhi789/text2sql-sqlite-clean | parthasaradhi789/text2sql-qwen2.5-coder-1.5b-lora | parthasaradhi789/text2sql-qwen2.5-coder-1.5b-GGUF | repo 1d71449


## 4. Load the 4-bit base model

In [6]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CFG["base_model"],
    max_seq_length=CFG["max_seq_length"],
    dtype=None,            # picks fp16 on a T4
    load_in_4bit=True,
)
print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.4: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
GPU memory in use: 1.15 GB


## 5. Add LoRA adapters

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=CFG["lora_dropout"],
    target_modules=CFG["target_modules"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CFG["seed"],
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} of {total:,} ({100 * trainable / total:.2f}%)")

Unsloth 2026.11.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
trainable: 18,464,768 of 907,081,216 (2.04%)


## 6. Data
Each row already has the final chat messages, built by the same `build_messages` the eval and the app use.

In [8]:
from datasets import load_dataset

train_ds = load_dataset(DATASET_REPO, "train", split="train")
val_ds = load_dataset(DATASET_REPO, "validation", split="validation")

def to_text(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False)}

train_text = train_ds.map(to_text)
val_text = val_ds.map(to_text)

lengths = [len(tokenizer(t)["input_ids"]) for t in train_text["text"]]
print(f"{len(train_text)} train / {len(val_text)} val rows; tokens max {max(lengths)}, "
      f"mean {sum(lengths) / len(lengths):.0f}")
assert max(lengths) <= CFG["max_seq_length"]
print(train_text[0]["text"])

README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

validation.jsonl: 0.00B [00:00, ?B/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/9500 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

9500 train / 300 val rows; tokens max 678, mean 219
<|im_start|>system
You are a SQLite expert. Given a database schema and a question, write one SQLite SELECT query that answers the question. Output only the SQL query.<|im_end|>
<|im_start|>user
Schema:
CREATE TABLE menu_items (item VARCHAR(50), type VARCHAR(15), cost DECIMAL(10,2));
-- Vegan Pizza, Vegan, 12
-- Vegan Pasta, Vegan, 15
-- VIEW total_menu_items
-- VIEW vegan_menu_items

Question: What is the percentage of vegan dishes in the menu?<|im_end|>
<|im_start|>assistant
SELECT 100.0 * COUNT(*) / (SELECT num_items FROM total_menu_items) as percentage FROM vegan_menu_items<|im_end|>



## 7. Baseline before training
Greedy-generate SQL for the 300 validation questions, run it in SQLite and score it with the same rules as the eval harness. Batched generation can differ a little from one-at-a-time serving, so this is a quick check. The final numbers come from the eval harness on the GGUF.

In [9]:
from text2sql.prompt import extract_sql
from text2sql.schema import load_context
from text2sql.sqlexec import results_match, run_select

tokenizer.padding_side = "left"

def generate_sql(rows, batch_size=16, max_new_tokens=256):
    outs = []
    for i in range(0, len(rows), batch_size):
        prompts = [tokenizer.apply_chat_template(r["messages"][:2], tokenize=False,
                                                 add_generation_prompt=True)
                   for r in rows[i:i + batch_size]]
        enc = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
        outs += tokenizer.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

def execution_accuracy(rows, raw_outputs):
    hits, errors = [], 0
    for row, out in zip(rows, raw_outputs):
        meta = row["meta"]
        conn = load_context(meta["context_sqlite"])
        gold = run_select(conn, meta["gold_sql"])
        pred = run_select(conn, extract_sql(out))
        conn.close()
        if not pred.ok:
            errors += 1
        hits.append(pred.ok and gold.ok and results_match(pred.rows, gold.rows, meta["order_matters"]))
    return sum(hits) / len(hits), errors, hits

val_rows = list(val_ds)
FastLanguageModel.for_inference(model)
base_out = generate_sql(val_rows)
base_acc, base_errors, base_hits = execution_accuracy(val_rows, base_out)
print(f"BASE: execution accuracy {base_acc:.1%} on {len(val_rows)} val items, "
      f"{base_errors} did not execute")
for row, out in list(zip(val_rows, base_out))[:3]:
    print("\nQ:", row["messages"][1]["content"].split("Question:")[-1].strip())
    print("gold:", row["meta"]["gold_sql"]); print("base:", extract_sql(out))

BASE: execution accuracy 57.0% on 300 val items, 43 did not execute

Q: What is the total production of oil and gas in the North Sea for the year 2020?
gold: SELECT SUM(production_volume) FROM production WHERE location = 'North Sea' AND production_type IN ('Oil', 'Gas') AND production_date BETWEEN '2020-01-01' AND '2020-12-31'
base: SELECT SUM(CASE WHEN production_type = 'Oil' THEN production_volume ELSE 0 END) + SUM(CASE WHEN production_type = 'Gas' THEN production_volume ELSE 0 END) AS total_production
FROM production
WHERE location = 'North Sea' AND YEAR(production_date) = 2020

Q: What is the total amount spent on raw materials for the entire year of 2020 and 2021?
gold: SELECT SUM(amount) FROM expenses WHERE date BETWEEN '2020-01-01' AND '2021-12-31' AND category = 'textile'
base: SELECT SUM(amount) FROM expenses WHERE YEAR(date) IN (2020, 2021) AND category = 'raw material'

Q: How many different animals were observed in the 'arctic_animal_sightings' table for each observer?
gold

## 8. Trainer

In [10]:
from trl import SFTConfig, SFTTrainer

FastLanguageModel.for_training(model)

run_name = f"qwen2.5c-1.5b-qlora-r{CFG['lora_r']}-{GIT_SHA}"
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=val_text,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=CFG["max_seq_length"],
        packing=False,
        per_device_train_batch_size=CFG["batch_size"],
        gradient_accumulation_steps=CFG["grad_accum"],
        num_train_epochs=CFG["epochs"],
        learning_rate=CFG["learning_rate"],
        lr_scheduler_type=CFG["lr_scheduler"],
        warmup_ratio=CFG["warmup_ratio"],
        optim=CFG["optimizer"],
        weight_decay=CFG["weight_decay"],
        fp16=True, bf16=False,                       # a T4 has no bf16
        logging_steps=10,
        eval_strategy="steps", eval_steps=CFG["eval_steps"],
        save_strategy="steps", save_steps=CFG["save_steps"], save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="eval_loss",
        push_to_hub=True, hub_model_id=OUT_REPO_LORA, hub_strategy="checkpoint",
        output_dir="outputs",
        seed=CFG["seed"],
        report_to="wandb", run_name=run_name,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/9500 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/300 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


## 9. Train on the answer only
Otherwise the model also learns to predict the schema and question. The check decodes the tokens that still count towards the loss: it should be the SQL plus `<|im_end|>`.

In [11]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
labels = trainer.train_dataset[0]["labels"]
target = tokenizer.decode([t for t in labels if t != -100])
expected = train_ds[0]["messages"][-1]["content"] + "<|im_end|>"
print(repr(target))
assert target.strip() == expected, "loss mask is wrong, don't train"

Map (num_proc=2):   0%|          | 0/9500 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/300 [00:00<?, ? examples/s]

'SELECT 100.0 * COUNT(*) / (SELECT num_items FROM total_menu_items) as percentage FROM vegan_menu_items<|im_end|>'


## 10. Time projection
Prints a projection for the whole run after 20 steps. If it says more than ~50 minutes, set `max_steps` in cell 8.

In [12]:
import time
from transformers import TrainerCallback

class Projection(TrainerCallback):
    def on_train_begin(self, args, state, control, **kw):
        self.t0 = time.time()
    def on_step_end(self, args, state, control, **kw):
        if state.global_step == 20:
            per_step = (time.time() - self.t0) / 20
            print(f"\n>>> {per_step:.2f} s/step -> {per_step * state.max_steps / 60:.0f} min "
                  f"for {state.max_steps} steps")

trainer.add_callback(Projection())

## 11. Train
Checkpoints go to the Hub every 200 steps. After a disconnect, re-run cells 0-10 and call `trainer.train(resume_from_checkpoint=True)`.

In [13]:
import wandb

wandb.init(project="text2sql-specialist", name=run_name, config={
    **CFG, "dataset": DATASET_REPO, "git_sha": GIT_SHA, "gpu": gpu,
    "trainable_params": trainable, "baseline_val_ex": base_acc,
})
stats = trainer.train()
print(stats.metrics)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: parthasaradhimunakala (parthasaradhimunakala-parthasaradhi-site) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,500 | Num Epochs = 1 | Total steps = 594
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,0.266200,0.196343
200,0.229900,0.182242
300,0.226500,0.173457
400,0.193900,0.169230
500,0.183900,0.166247



>>> 2.85 s/step -> 28 min for 594 steps


Filter:   0%|          | 0/300 [00:00<?, ? examples/s]

{'train_runtime': 1546.239, 'train_samples_per_second': 6.144, 'train_steps_per_second': 0.384, 'total_flos': 1.6617303874077696e+16, 'train_loss': 0.22175548764993044, 'epoch': 1.0}


## 12. Did it help?
Same 300 questions and scoring as cell 7.

In [14]:
FastLanguageModel.for_inference(model)
ft_out = generate_sql(val_rows)
ft_acc, ft_errors, ft_hits = execution_accuracy(val_rows, ft_out)
print(f"BASE        {base_acc:.1%}  ({base_errors} did not execute)")
print(f"FINE-TUNED  {ft_acc:.1%}  ({ft_errors} did not execute)   n={len(val_rows)}")
wandb.run.summary.update({"val_ex_base": base_acc, "val_ex_finetuned": ft_acc})

table = wandb.Table(columns=["question", "gold", "base", "finetuned", "base_ok", "ft_ok"])
for row, b, f, bh, fh in list(zip(val_rows, base_out, ft_out, base_hits, ft_hits))[:20]:
    table.add_data(row["messages"][1]["content"].split("Question:")[-1].strip(),
                   row["meta"]["gold_sql"], extract_sql(b), extract_sql(f), bh, fh)
wandb.log({"val_predictions": table})

BASE        57.0%  (43 did not execute)
FINE-TUNED  69.3%  (11 did not execute)   n=300


## 13. Save the LoRA adapter

In [15]:
from huggingface_hub import ModelCard

model.push_to_hub(OUT_REPO_LORA, token=os.environ["HF_TOKEN"])
tokenizer.push_to_hub(OUT_REPO_LORA, token=os.environ["HF_TOKEN"])
card = f'''---
base_model: Qwen/Qwen2.5-Coder-1.5B-Instruct
library_name: peft
license: apache-2.0
datasets: [{DATASET_REPO}]
tags: [text-to-sql, sqlite, qlora, unsloth]
---
# Qwen2.5-Coder-1.5B text-to-SQL (SQLite) LoRA

QLoRA adapter (r={CFG["lora_r"]}, alpha={CFG["lora_alpha"]}, all linear layers) trained for
{CFG["epochs"]} epoch on [{DATASET_REPO}](https://huggingface.co/datasets/{DATASET_REPO}).

Quick check on 300 validation questions from unseen domains (execution accuracy, notebook
batched greedy decoding): base {base_acc:.1%} -> fine-tuned {ft_acc:.1%}.
Final numbers come from the evaluation harness in the code repository ({REPO_URL}).
Git commit used for training: `{GIT_SHA}`. Training run: {wandb.run.url}
'''
ModelCard(card).push_to_hub(OUT_REPO_LORA, token=os.environ["HF_TOKEN"])

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  54%|#####4    | 39.9MB / 73.9MB            

Saved model to https://huggingface.co/parthasaradhi789/text2sql-qwen2.5-coder-1.5b-lora


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mptbpceqqh/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

CommitInfo(commit_url='https://huggingface.co/parthasaradhi789/text2sql-qwen2.5-coder-1.5b-lora/commit/dfd1137788624f1d49bdff3c59d079a5ef3d25ba', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='dfd1137788624f1d49bdff3c59d079a5ef3d25ba', pr_url=None, repo_url=RepoUrl('https://huggingface.co/parthasaradhi789/text2sql-qwen2.5-coder-1.5b-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='parthasaradhi789/text2sql-qwen2.5-coder-1.5b-lora'), pr_revision=None, pr_num=None)

## 14. Export GGUF
Slow (about ten minutes) because llama.cpp is compiled first.

In [16]:
model.push_to_hub_gguf(
    OUT_REPO_GGUF, tokenizer,
    quantization_method=["q4_k_m", "q8_0", "f16"],
    token=os.environ["HF_TOKEN"],
)

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/765 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:41<00:00, 41.29s/it]


... (6 outputs left out)

  ...r-1.5B-Instruct.Q8_0.gguf:   0%|          |  548kB / 1.65GB            

  ...er-1.5B-Instruct.F16.gguf:   0%|          |  548kB / 3.09GB            

Unsloth: Successfully uploaded GGUF to https://huggingface.co/parthasaradhi789/text2sql-qwen2.5-coder-1.5b-GGUF
Unsloth: Cleaning up temporary files...


'parthasaradhi789/text2sql-qwen2.5-coder-1.5b-GGUF'

## 15. Finish

In [17]:
run_url = wandb.run.url
wandb.finish()
print("adapter :", f"https://huggingface.co/{OUT_REPO_LORA}")
print("gguf    :", f"https://huggingface.co/{OUT_REPO_GGUF}")
print("dataset :", f"https://huggingface.co/datasets/{DATASET_REPO}")
print("wandb   :", run_url)

eval/loss,█▅▃▂▁
eval/runtime,██▂▁▃
eval/samples_per_second,▁▁▇█▆
eval/steps_per_second,▁▁▇█▆
train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train/grad_norm,▄▃▂▂▂█▃▃▃▂▃▄▃▄▃▂▃▂▃▂▅▂▄▃▃▁▂▃▂▃▂▂▂▁▂▂▅▁▂▂
train/learning_rate,▄████████▇▇▇▇▇▆▆▆▆▆▅▅▅▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁▁▁▁
train/loss,█▆▄▅▄▃▄▃▃▂▂▄▄▃▃▂▂▁▂▂▂▂▂▂▂▂▁▂▂▁▁▁▁▁▂▂▁▁▂▁
eval/loss,0.16625
eval/runtime,12.8484


adapter : https://huggingface.co/parthasaradhi789/text2sql-qwen2.5-coder-1.5b-lora
gguf    : https://huggingface.co/parthasaradhi789/text2sql-qwen2.5-coder-1.5b-GGUF
dataset : https://huggingface.co/datasets/parthasaradhi789/text2sql-sqlite-clean
wandb   : https://wandb.ai/parthasaradhimunakala-parthasaradhi-site/text2sql-specialist/runs/379av0n3
